In [2]:
import glob
import os

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import napari

import skimage
from scipy import ndimage as ndi
from skimage import measure, segmentation, filters
from skimage.measure import regionprops
from skimage.util import compare_images

### Importing and loading

In [ ]:
# Import images

image_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/Max_projections_seperate/Halo'
image_path = os.path.join(image_dir,'*.tif') 
image_files = glob.glob(image_path)

image_files_sorted = np.sort(image_files)

image_files_sorted

In [ ]:
# Import images perturbation

image_puro_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Puro10min/Max_projections_seperate/Halo'
image_puro_path = os.path.join(image_puro_dir,'*.tif') 
image_puro_files = glob.glob(image_puro_path)

image_puro_files_sorted = np.sort(image_puro_files)

image_puro_files_sorted

In [4]:
# Read images into list

images = []

for file in image_files_sorted:
    image = imread(file)
    images.append(image)

In [5]:
# Read perturbation images into list

images_puro = []

for file in image_puro_files_sorted:
    image_puro = imread(file)
    images_puro.append(image_puro)

In [ ]:
plt.imshow(images[-1])

In [ ]:
# Import masks

mask_cell_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/ROIs/Whole_cell'
mask_cell_path = os.path.join(mask_cell_dir,'*.tif') 
mask_cell_files = glob.glob(mask_cell_path)

mask_nucleus_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/ROIs/Nucleus'
mask_nucleus_path = os.path.join(mask_nucleus_dir,'*.tif') 
mask_nucleus_files = glob.glob(mask_nucleus_path)

# Sorts images in list alphabetically

mask_cell_sorted = np.sort(mask_cell_files)
mask_nucleus_sorted = np.sort(mask_nucleus_files)

mask_nucleus_sorted

In [ ]:
# Import masks puro

mask_cell_puro_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Puro10min/ROIs/Whole_cell'
mask_cell_puro_path = os.path.join(mask_cell_puro_dir,'*.tif') 
mask_cell_puro_files = glob.glob(mask_cell_puro_path)

mask_nucleus_puro_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D2/Unperturbed/ROIs/Nucleus'
mask_nucleus_puro_path = os.path.join(mask_nucleus_puro_dir,'*.tif') 
mask_nucleus_puro_files = glob.glob(mask_nucleus_puro_path)

# Sorts images in list alphabetically

mask_cell_puro_sorted = np.sort(mask_cell_puro_files)
mask_nucleus_puro_sorted = np.sort(mask_nucleus_puro_files)

mask_nucleus_puro_sorted


In [ ]:
print(len(image_files_sorted))
print(len(mask_cell_sorted))
print(len(mask_nucleus_sorted))
print(len(image_puro_files_sorted))
print(len(mask_cell_puro_sorted))
print(len(mask_nucleus_puro_sorted))

In [8]:
# Read images into list

mask_cell = []
mask_nucleus = []

for file in mask_cell_sorted:
    image = imread(file)
    mask_cell.append(image)

for file in mask_nucleus_sorted:
    image = imread(file)
    mask_nucleus.append(image)

In [ ]:
# Read images into list

mask_cell_puro = []
mask_nucleus_puro = []

for file in mask_cell_puro_sorted:
    image_puro = imread(file)
    mask_cell_puro.append(image_puro)

for file in mask_nucleus_puro_sorted:
    image_puro = imread(file)
    mask_nucleus_puro.append(image_puro)

In [ ]:
np.unique(mask_cell_puro[3])

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(mask_cell[-2])
ax[1].imshow(mask_nucleus[-2])

### Watershedding from nucleus to separate cell masks

In [11]:
# Watershed using the nucleus as seed to separate the overlapping cell masks

mask_label=[]
distance_maps = []

for cell, nucleus in zip(mask_cell, mask_nucleus):
    
    # 1. Smooth the distance transform of the cell mask
    distance_map = ndi.distance_transform_edt(cell)
    smoothed_distance_map = filters.gaussian(distance_map, sigma=1)
    distance_maps.append(smoothed_distance_map)

    # 2. Use the nuclei labels as markers
    markers = measure.label(nucleus)

    # 3. Apply the watershed algorithm
    labels = segmentation.watershed(-smoothed_distance_map, markers, mask=cell)
    mask_label.append(labels)

In [ ]:
# Plotting the results including the distance map
fig, ax = plt.subplots(1, 4, figsize=(20, 5))

ax[0].imshow(mask_cell[-2], cmap='gray')
ax[0].set_title('Cell Mask')

ax[1].imshow(mask_nucleus[-2], cmap='gray')
ax[1].set_title('Nucleus Mask')

ax[2].imshow(distance_maps[-2], cmap='gray')
ax[2].set_title('Distance Map')

ax[3].imshow(mask_label[-2])
ax[3].set_title('Separated Cell Labels')

for a in ax:
    a.axis('off')

plt.show()

In [ ]:
np.unique(mask_label[-1])

In [17]:
# Watershed using the nucleus as seed to separate the overlapping cell masks - perturbation

mask_label_puro=[]
distance_maps_puro = []

for cell, nucleus in zip(mask_cell_puro, mask_nucleus_puro):
    
    # 1. Smooth the distance transform of the cell mask
    distance_map = ndi.distance_transform_edt(cell)
    smoothed_distance_map = filters.gaussian(distance_map, sigma=1)
    distance_maps_puro.append(smoothed_distance_map)

    # 2. Use the nuclei labels as markers
    markers = measure.label(nucleus)

    # 3. Apply the watershed algorithm
    labels = segmentation.watershed(-smoothed_distance_map, markers, mask=cell)
    mask_label_puro.append(labels)


In [ ]:
# Plotting only the separated cell labels
plt.figure(figsize=(8, 8))
plt.imshow(mask_label_puro[3])
plt.title('Separated Cell Labels')
plt.axis('off')
plt.show()

In [ ]:
print(np.unique(mask_label_puro[3]))
print(np.unique(mask_nucleus_puro[3]))

### Creating cytoplasm mask

In [14]:
# To match intensity values (= labels) of ROIs in nucleus image to intensity values of ROIs in whole cell image

for nucleus_image, cell_image in zip(mask_nucleus, mask_label):
    # Create a dictionary to store coordinates and corresponding intensity values from mask_cell
    coordinates_intensity = {}

    # Store coordinates and intensity values from cell_image in the dictionary
    for cell_label in regionprops(cell_image):
        for coord in cell_label.coords:
            coordinates_intensity[tuple(coord)] = cell_image[coord[0], coord[1]]

    # Update nucleus_image based on the mapped coordinates and intensity values
    for nucleus_label in regionprops(nucleus_image):
        for coord in nucleus_label.coords:
            if tuple(coord) in coordinates_intensity:
                nucleus_image[coord[0], coord[1]] = coordinates_intensity[tuple(coord)]

In [21]:
# To match intensity values (= labels) of ROIs in nucleus image to intensity values of ROIs in whole cell image - perturbation

for nucleus_image, cell_image in zip(mask_nucleus_puro, mask_label_puro):
    # Create a dictionary to store coordinates and corresponding intensity values from mask_cell
    coordinates_intensity = {}

    # Store coordinates and intensity values from cell_image in the dictionary
    for cell_label in regionprops(cell_image):
        for coord in cell_label.coords:
            coordinates_intensity[tuple(coord)] = cell_image[coord[0], coord[1]]

    # Update nucleus_image based on the mapped coordinates and intensity values
    for nucleus_label in regionprops(nucleus_image):
        for coord in nucleus_label.coords:
            if tuple(coord) in coordinates_intensity:
                nucleus_image[coord[0], coord[1]] = coordinates_intensity[tuple(coord)]

In [ ]:
print(np.unique(mask_label[1]))
print(np.unique(mask_nucleus[1]))

In [ ]:
# Plot results of label matching

fig, ax = plt.subplots(1, 4, figsize = (16,4))
ax[0].imshow(mask_label[-1])
ax[1].imshow(mask_nucleus[-1])
ax[2].imshow(mask_label[-2])
ax[3].imshow(mask_nucleus[-2])

In [ ]:
# Substract nucleus mask from cell mask and store as binary cytoplasm mask

def subtract_nucleus_from_cell(cell_mask, nucleus_mask):
    """
    Subtract the nucleus mask from the cell mask and return the cytoplasm mask.
    """
    # Create a copy of the cell mask to modify
    cytoplasm_mask = cell_mask.copy()
    
    # Set the areas where nucleus label is present to 0 in the cell mask
    cytoplasm_mask[nucleus_mask > 0] = 0
    
    return cytoplasm_mask

# List to store cytoplasm masks
mask_cytoplasm = []

for cell, nucleus in zip(mask_label, mask_nucleus):
    # Subtract nucleus mask from cell mask
    cytoplasm_mask = subtract_nucleus_from_cell(cell, nucleus)
    
    mask_cytoplasm.append(cytoplasm_mask)

# List to store cytoplasm masks - perturbation
mask_cytoplasm_puro = []

for cell, nucleus in zip(mask_label_puro, mask_nucleus_puro):
    # Subtract nucleus mask from cell mask
    cytoplasm_mask = subtract_nucleus_from_cell(cell, nucleus)
    
    mask_cytoplasm_puro.append(cytoplasm_mask)

# Check results
for i, cytoplasm in enumerate(mask_cytoplasm):
    print(f"Unique values in cytoplasm mask {i}: {np.unique(cytoplasm)}")

# Check results - perturbation
for i, cytoplasm in enumerate(mask_cytoplasm_puro):
    print(f"Unique values in puro cytoplasm mask {i}: {np.unique(mask_cytoplasm_puro)}")


In [ ]:
# Plot binary cytoplasm mask

plt.imshow(mask_cytoplasm[4])

In [ ]:
# Plot binary cytoplasm mask

plt.imshow(mask_cytoplasm[6])

In [ ]:
print(np.unique(mask_cytoplasm_puro[4]))
print(np.unique(mask_cytoplasm[6]))

### Mask corrections with Napari for overlapping cells

Click pasteur pipette to pick a label. Use paint brush to draw a line to sperate two labels. Fill entire area with according label with fill bucket option. To create background, select label 0 and use the paint brush to draw and bucket to fill up.

In [ ]:
# Create copies of cytoplasm masks for manipulation in Napari

mask_cytoplasm_copy = mask_cytoplasm.copy()
plt.imshow(mask_cytoplasm_copy[-1])

In [ ]:
# Initializes Napari viewer

viewer = napari.Viewer()

In [34]:
# Adds images to Napari viewer

for i, img in enumerate(images):
    layer_name = f'image_{i}'  # Dynamically name each layer
    viewer.add_image(img, name=layer_name)

In [28]:
# Adds masks to Napari viewer

for i, img in enumerate(mask_cytoplasm_copy):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer.add_labels(img, name=layer_name, opacity=0.2)

In [ ]:
# Create copies of cytoplasm masks for manipulation in Napari

mask_cytoplasm_puro_copy = mask_cytoplasm_puro.copy()
plt.imshow(mask_cytoplasm_puro_copy[0])

In [ ]:
plt.imshow(mask_cytoplasm_copy[1])

In [34]:
# Initializes Napari viewer

viewer2 = napari.Viewer()

In [35]:
# Adds images to Napari viewer

for i, img in enumerate(images_puro):
    layer_name = f'image_{i}'  # Dynamically name each layer
    viewer2.add_image(img, name=layer_name)

In [36]:
# Adds masks to Napari viewer

for i, img in enumerate(mask_cytoplasm_puro_copy):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer2.add_labels(img, name=layer_name, opacity=0.2)

In [ ]:
plt.imshow(mask_cytoplasm_puro_copy[3])

### Saving finished masks

Only save mask that have more than one label, i.e. more than background.

When saving, then the "halo" part from the file name should be deleted, so that the masks have the same name as the original images except the _ROIs at the end (for spot detection).

After saving, images where no masks were created have to be removed from the Selected_for_analysis folder.

In [23]:
# Save cytoplasm masks

mask_cytoplasm_dir = r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/ROIs/Cytoplasm'


In [ ]:
# Save mask images
#for img, tiff_file in zip(mask_cytoplasm_copy, mask_cell_sorted):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    print(file_name)
    
    # Check if the image has more than just the background (label 0)
    if len(np.unique(img)) > 1:  # This checks for more than one unique value
        # Define the output file path
        output_file = os.path.join(mask_cytoplasm_dir, file_name + '.tif')
        print(output_file)
        
        # Save the manipulated image as a TIFF
        tf.imwrite(output_file, img)
    else:
        print(f"Image {file_name} has only background (label 0), not saving.")

In [ ]:
# Save mask images with removing last part of the file name

for img, tiff_file in zip(mask_cytoplasm_copy, mask_cell_sorted):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    # Remove the second-to-last part
    parts = file_name.split('_')
    if len(parts) > 2:  # Ensure there are enough parts to modify
        file_name = '_'.join(parts[:-2] + parts[-1:])
    
    print(file_name)
    
    # Check if the image has more than just the background (label 0)
    if len(np.unique(img)) > 1:  # This checks for more than one unique value
        # Define the output file path
        output_file = os.path.join(mask_cytoplasm_dir, file_name + '.tif')
        print(output_file)
        
        # Save the manipulated image as a TIFF
        tf.imwrite(output_file, img)
    else:
        print(f"Image {file_name} has only background (label 0), not saving.")


In [81]:
# Save cytoplasm masks - perturbation

#mask_cytoplasm_puro_dir = r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Puro10min/ROIs/Cytoplasm'


In [ ]:
# Save mask images
#for img, tiff_file in zip(mask_cytoplasm_puro_copy, mask_cell_puro_sorted):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    print(file_name)
    
    # Check if the image has more than just the background (label 0)
    if len(np.unique(img)) > 1:  # This checks for more than one unique value
        # Define the output file path
        output_file = os.path.join(mask_cytoplasm_puro_dir, file_name + '.tif')
        print(output_file)
        
        # Save the manipulated image as a TIFF
        tf.imwrite(output_file, img)
    else:
        print(f"Image {file_name} has only background (label 0), not saving.")

In [ ]:
# Save mask images with removing last part of the file name - perturbation

for img, tiff_file in zip(mask_cytoplasm_puro_copy, mask_cell_puro_sorted):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    # Remove the second-to-last part
    parts = file_name.split('_')
    if len(parts) > 2:  # Ensure there are enough parts to modify
        file_name = '_'.join(parts[:-2] + parts[-1:])
    
    print(file_name)
    
    # Check if the image has more than just the background (label 0)
    if len(np.unique(img)) > 1:  # This checks for more than one unique value
        # Define the output file path
        output_file = os.path.join(mask_cytoplasm_puro_dir, file_name + '.tif')
        print(output_file)
        
        # Save the manipulated image as a TIFF
        tf.imwrite(output_file, img)
    else:
        print(f"Image {file_name} has only background (label 0), not saving.")


### Drawing nuclei

In [25]:
mask_nucleus_puro_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/ROIs/Nucleus'
mask_nucleus_puro_path = os.path.join(mask_nucleus_puro_dir,'*.tif') 
mask_nucleus_puro_files = glob.glob(mask_nucleus_puro_path)


mask_nucleus = []

for file in mask_nucleus_sorted:
    image = imread(file)
    mask_nucleus.append(image)

In [ ]:
mask_nucleus_sorted

In [ ]:
drawn_nuclei = [mask_nucleus[-2],mask_nucleus[1],mask_nucleus[6]]
type(drawn_nuclei)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(drawn_nuclei[0], cmap='gray')
ax[0].set_title('Nucleus missing')

ax[1].imshow(drawn_nuclei[2], cmap='gray')
ax[1].set_title('Nucleus empty')

In [258]:
images_missing_nuclei = [images[23],images[11]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(images_missing_nuclei[0], cmap='gray')
ax[0].set_title('Nucleus missing')

ax[1].imshow(images_missing_nuclei[1], cmap='gray')
ax[1].set_title('Nucleus empty')

In [219]:
# Adds images to Napari viewer

for i, img in enumerate(images_missing_nuclei):
    layer_name = f'image_{i}'  # Dynamically name each layer
    viewer.add_image(img, name=layer_name)

In [37]:
# Adds masks to Napari viewer

for i, img in enumerate(drawn_nuclei):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer.add_labels(img, name=layer_name, opacity=0.2)

In [38]:
# Save cytoplasm masks - perturbation

mask_missing_nuclei_dir = r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/ROIs/Nucleus/Missing_nuclei'


In [ ]:
# Save mask images with removing last part of the file name

for img, tiff_file in zip(drawn_nuclei, mask_cell_sorted):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    # Remove the second-to-last part
    parts = file_name.split('_')
    if len(parts) > 2:  # Ensure there are enough parts to modify
        file_name = '_'.join(parts[:-2] + parts[-1:])
    
    print(file_name)
    
    # Check if the image has more than just the background (label 0)
    if len(np.unique(img)) > 1:  # This checks for more than one unique value
        # Define the output file path
        output_file = os.path.join(mask_missing_nuclei_dir, file_name + '.tif')
        print(output_file)
        
        # Save the manipulated image as a TIFF
        tf.imwrite(output_file, img)
    else:
        print(f"Image {file_name} has only background (label 0), not saving.")


: 